# Demographic Prediction

Amazon does not collect demographic information from users, beyond account information. This means we must be able to deduce user demographics to use in our prediction model.

First, I will encode our data such that demographics will be label encoded, rather than one-hot encoded.  

In [ ]:
# %pip install pandas matplotlib seaborn scikit-learn tensorflow

In [1]:
import pandas as pd
data = pd.read_csv('/workspaces/group-project-bas-team/data/data_unencoded.csv')

In [2]:
data[['order_month', 'order_day', 'order_year']] = data['Order Date'].str.split('/', expand=True)

data['Order Date'] = pd.to_datetime(data['Order Date'])
data = data.sort_values(by=['Survey ResponseID', 'Order Date'])
data['Purchase_Order'] = data.groupby('Survey ResponseID')['Order Date'].rank(method='first').astype(int)

data['Days_Since_Last_Purchase'] = data.groupby('Survey ResponseID')['Order Date'].diff().dt.days.fillna(0)

data = data.drop('Order Date', axis=1)

data[['order_month', 'order_day', 'order_year']] = data[['order_month', 'order_day', 'order_year']].apply(pd.to_numeric)

data.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Data columns (total 34 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Purchase Price Per Unit   157026 non-null  float64
 1   Quantity                  157026 non-null  int64  
 2   Shipping Address State    157026 non-null  str    
 3   Title                     157026 non-null  str    
 4   ASIN/ISBN (Product Code)  157026 non-null  str    
 5   Category                  157026 non-null  str    
 6   Survey ResponseID         157026 non-null  str    
 7   age                       157026 non-null  str    
 8   hispanic                  157026 non-null  str    
 9   race                      157026 non-null  str    
 10  education                 157026 non-null  str    
 11  income                    157026 non-null  str    
 12  gender                    157026 non-null  str    
 13  sexual-orientation        157026 non-null  str    
 14 

In [3]:
cols_to_drop = [
    'sell-YOUR-data',
    'sell-consumer-data',
    'small-biz-use',
    'census-use',
    'research-society'
]

data = data.drop(columns=cols_to_drop)

In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

cols = ['Survey ResponseID', 'Title', 'ASIN/ISBN (Product Code)', 'Category', 'age', 
        'hispanic', 'race', 'education', 'income', 'gender',
        'sexual-orientation', 'state', 'howmany', 'hh-size', 'how-oft',
        'cigarettes', 'marijuana', 'alcohol', 'diabetes', 'wheelchair', 'life-changes']

for col in cols:
    data[col] = le.fit_transform(data[col].astype(str))

In [5]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

data_one_hot = encoder.fit_transform(data[['Shipping Address State']])
df_one_hot = pd.DataFrame(data_one_hot, columns=encoder\
                          .get_feature_names_out(['Shipping Address State']))

df_one_hot = df_one_hot.drop(columns=['Shipping Address State_HI']) # removing one column to avoid redundancy

In [6]:
data = pd.concat([data, df_one_hot], axis=1)
data = data.drop(columns=['Shipping Address State'])
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Data columns (total 77 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Purchase Price Per Unit    157026 non-null  float64
 1   Quantity                   157026 non-null  int64  
 2   Title                      157026 non-null  int64  
 3   ASIN/ISBN (Product Code)   157026 non-null  int64  
 4   Category                   157026 non-null  int64  
 5   Survey ResponseID          157026 non-null  int64  
 6   age                        157026 non-null  int64  
 7   hispanic                   157026 non-null  int64  
 8   race                       157026 non-null  int64  
 9   education                  157026 non-null  int64  
 10  income                     157026 non-null  int64  
 11  gender                     157026 non-null  int64  
 12  sexual-orientation         157026 non-null  int64  
 13  state                      157026 non-nu

In [7]:
data.columns

Index(['Purchase Price Per Unit', 'Quantity', 'Title',
       'ASIN/ISBN (Product Code)', 'Category', 'Survey ResponseID', 'age',
       'hispanic', 'race', 'education', 'income', 'gender',
       'sexual-orientation', 'state', 'howmany', 'hh-size', 'how-oft',
       'cigarettes', 'marijuana', 'alcohol', 'diabetes', 'wheelchair',
       'life-changes', 'order_month', 'order_day', 'order_year',
       'Purchase_Order', 'Days_Since_Last_Purchase',
       'Shipping Address State_AK', 'Shipping Address State_AL',
       'Shipping Address State_AR', 'Shipping Address State_AZ',
       'Shipping Address State_CA', 'Shipping Address State_CO',
       'Shipping Address State_CT', 'Shipping Address State_DC',
       'Shipping Address State_DE', 'Shipping Address State_FL',
       'Shipping Address State_GA', 'Shipping Address State_IA',
       'Shipping Address State_ID', 'Shipping Address State_IL',
       'Shipping Address State_IN', 'Shipping Address State_KS',
       'Shipping Address State

Next, I will split the data into a train and test set

In [8]:
train = data[data['order_year']<=2021]
test = data[data['order_year']>2021]

For X_train and X_test, we only want non-demographic variables.
For y_train and y_test, we only want demographic variables.

In [9]:
from sklearn.preprocessing import StandardScaler

X_train = train.drop(columns = ['age', 'hispanic', 'race', 'education', 'income', 'gender',
                    'sexual-orientation', 'state', 'howmany', 'hh-size', 'how-oft',
                    'cigarettes', 'marijuana', 'alcohol', 'diabetes', 'wheelchair',
                    'life-changes'])
y_train = train[['age', 'hispanic', 'race', 'education', 'income', 'gender',
                 'sexual-orientation', 'state', 'howmany', 'hh-size', 'how-oft',
                 'cigarettes', 'marijuana', 'alcohol', 'diabetes', 'wheelchair',
                 'life-changes']]

X_test = test.drop(columns = ['age', 'hispanic', 'race', 'education', 'income', 'gender',
                    'sexual-orientation', 'state', 'howmany', 'hh-size', 'how-oft',
                    'cigarettes', 'marijuana', 'alcohol', 'diabetes', 'wheelchair',
                    'life-changes'])
y_test = test[['age', 'hispanic', 'race', 'education', 'income', 'gender',
                 'sexual-orientation', 'state', 'howmany', 'hh-size', 'how-oft',
                 'cigarettes', 'marijuana', 'alcohol', 'diabetes', 'wheelchair',
                 'life-changes']]

X_train_scaled = StandardScaler().fit_transform(X_train)
X_test_scaled = StandardScaler().fit_transform(X_test)

Next, I will create a neural network to predict each demographic variable

In [10]:
import tensorflow as tf

I0000 00:00:1777767885.963828   26163 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777767887.834579   26163 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777767894.960831   26163 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [45]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

inputs = tf.keras.layers.Input(shape=(60,))

hidden = tf.keras.layers.Dense(64, activation='relu')(inputs)
hidden = tf.keras.layers.Dense(32, activation='relu')(hidden)

age = tf.keras.layers.Dense(6, activation='softmax', name='age')(hidden)
hispanic = tf.keras.layers.Dense(2, activation='softmax', name='hispanic')(hidden)
race = tf.keras.layers.Dense(14, activation='softmax', name='race')(hidden)
education = tf.keras.layers.Dense(6, activation='softmax', name='education')(hidden)
income = tf.keras.layers.Dense(7, activation='softmax', name='income')(hidden)
gender = tf.keras.layers.Dense(4, activation='softmax', name='gender')(hidden)
sexual_orientation = tf.keras.layers.Dense(3, activation='softmax', name='sexual-orientation')(hidden)
state = tf.keras.layers.Dense(45, activation='softmax', name='state')(hidden)
howmany = tf.keras.layers.Dense(4, activation='softmax', name='howmany')(hidden)
hh_size = tf.keras.layers.Dense(4, activation='softmax', name='hh-size')(hidden)
how_oft = tf.keras.layers.Dense(3, activation='softmax', name='how-oft')(hidden)
cigarettes = tf.keras.layers.Dense(4, activation='softmax', name='cigarettes')(hidden)
marijuana = tf.keras.layers.Dense(4, activation='softmax', name='marijuana')(hidden)
alcohol = tf.keras.layers.Dense(4, activation='softmax', name='alcohol')(hidden)
diabetes = tf.keras.layers.Dense(3, activation='softmax', name='diabetes')(hidden)
wheelchair = tf.keras.layers.Dense(3, activation='softmax', name='wheelchair')(hidden)
life_changes = tf.keras.layers.Dense(18, activation='softmax', name='life-changes')(hidden)


model = tf.keras.Model(inputs=inputs, outputs=[age, hispanic, race, education, income, gender, sexual_orientation,
                                               state, howmany, hh_size, how_oft, cigarettes, marijuana, alcohol,
                                               diabetes, wheelchair, life_changes])

model.compile(
    optimizer='sgd',
    loss= 'sparse_categorical_crossentropy',
    metrics={'age': 'accuracy',
             'hispanic': 'accuracy',
             'race': 'accuracy',
             'education': 'accuracy',
             'income': 'accuracy',
             'gender': 'accuracy',
             'sexual-orientation': 'accuracy',
             'state': 'accuracy',
             'howmany': 'accuracy',
             'hh-size': 'accuracy',
             'how-oft': 'accuracy',
             'cigarettes': 'accuracy',
             'marijuana': 'accuracy',
             'alcohol': 'accuracy',
             'diabetes': 'accuracy',
             'wheelchair': 'accuracy',
             'life-changes': 'accuracy'})

In [43]:
y_train_age = y_train['age']
y_train_hispanic = y_train['hispanic']
y_train_race = y_train['race']
y_train_education = y_train['education']
y_train_income = y_train['income']
y_train_gender = y_train['gender']
y_train_sexual_orientation = y_train['sexual-orientation']
y_train_state = y_train['state']
y_train_howmany = y_train['howmany']
y_train_hh_size = y_train['hh-size']
y_train_how_oft = y_train['how-oft']
y_train_cigarettes = y_train['cigarettes']
y_train_marijuana = y_train['marijuana']
y_train_alcohol = y_train['alcohol']
y_train_diabetes = y_train['diabetes']
y_train_wheelchair = y_train['wheelchair']
y_train_life_changes = y_train['life-changes']


In [47]:
history = model.fit(X_train_scaled, [y_train_age, y_train_hispanic, y_train_race, 
                                     y_train_education, y_train_income, y_train_gender,
                                     y_train_sexual_orientation, y_train_state, y_train_hh_size,
                                     y_train_howmany, y_train_how_oft, y_train_cigarettes, 
                                     y_train_marijuana, y_train_alcohol, y_train_diabetes,
                                     y_train_wheelchair, y_train_life_changes], epochs=30)

Epoch 1/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - age_accuracy: 0.5234 - age_loss: 1.1660 - alcohol_accuracy: 0.6875 - alcohol_loss: 0.5923 - cigarettes_accuracy: 0.8426 - cigarettes_loss: 0.3937 - diabetes_accuracy: 0.9271 - diabetes_loss: 0.2255 - education_accuracy: 0.6091 - education_loss: 0.8385 - gender_accuracy: 0.6970 - gender_loss: 0.5807 - hh-size_accuracy: 0.7130 - hh-size_loss: 0.6934 - hispanic_accuracy: 0.9306 - hispanic_loss: 0.1802 - how-oft_accuracy: 0.7254 - how-oft_loss: 0.6292 - howmany_accuracy: 0.6281 - howmany_loss: 0.8989 - income_accuracy: 0.5271 - income_loss: 1.2224 - life-changes_accuracy: 0.7574 - life-changes_loss: 0.7225 - loss: 10.0570 - marijuana_accuracy: 0.8182 - marijuana_loss: 0.4218 - race_accuracy: 0.8311 - race_loss: 0.5400 - sexual-orientation_accuracy: 0.8322 - sexual-orientation_loss: 0.4024 - state_accuracy: 0.9101 - state_loss: 0.5033 - wheelchair_accuracy: 0.9902 - wheelchair_loss: 0.0460
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s

In [48]:
print(f'age accuracy: {history.history['age_accuracy'][-1]}')
print(f'hispanic accuracy: {history.history['hispanic_accuracy'][-1]}')
print(f'race accuracy: {history.history['race_accuracy'][-1]}')
print(f'education accuracy: {history.history['education_accuracy'][-1]}')
print(f'income accuracy: {history.history['income_accuracy'][-1]}')
print(f'gender accuracy: {history.history['gender_accuracy'][-1]}')
print(f'sexual orientation accuracy: {history.history['sexual-orientation_accuracy'][-1]}')
print(f'state accuracy: {history.history['state_accuracy'][-1]}')
print(f'howmany accuracy: {history.history['howmany_accuracy'][-1]}')
print(f'hh-size accuracy: {history.history['hh-size_accuracy'][-1]}')
print(f'how-oft accuracy: {history.history['how-oft_accuracy'][-1]}')
print(f'cigarettes accuracy: {history.history['cigarettes_accuracy'][-1]}')
print(f'marijuana accuracy: {history.history['marijuana_accuracy'][-1]}')
print(f'alcohol accuracy: {history.history['alcohol_accuracy'][-1]}')
print(f'diabetes accuracy: {history.history['diabetes_accuracy'][-1]}')
print(f'wheelchair accuracy: {history.history['wheelchair_accuracy'][-1]}')
print(f'life changes accuracy: {history.history['life-changes_accuracy'][-1]}')

age accuracy: 0.8246359825134277
hispanic accuracy: 0.9710791707038879
race accuracy: 0.9278342127799988
education accuracy: 0.8546302318572998
income accuracy: 0.8093264698982239
gender accuracy: 0.8770753741264343
sexual orientation accuracy: 0.9249923229217529
state accuracy: 0.9531740546226501
howmany accuracy: 0.8339360356330872
hh-size accuracy: 0.8818177580833435
how-oft accuracy: 0.8675464987754822
cigarettes accuracy: 0.917927086353302
marijuana accuracy: 0.9322159290313721
alcohol accuracy: 0.8744621872901917
diabetes accuracy: 0.957186222076416
wheelchair accuracy: 0.9936650395393372
life changes accuracy: 0.8944700956344604


In [49]:
y_test_age = y_test['age']
y_test_hispanic = y_test['hispanic']
y_test_race = y_test['race']
y_test_education = y_test['education']
y_test_income = y_test['income']
y_test_gender = y_test['gender']
y_test_sexual_orientation = y_test['sexual-orientation']
y_test_state = y_test['state']
y_test_howmany = y_test['howmany']
y_test_hh_size = y_test['hh-size']
y_test_how_oft = y_test['how-oft']
y_test_cigarettes = y_test['cigarettes']
y_test_marijuana = y_test['marijuana']
y_test_alcohol = y_test['alcohol']
y_test_diabetes = y_test['diabetes']
y_test_wheelchair = y_test['wheelchair']
y_test_life_changes = y_test['life-changes']

In [63]:
test_results = model.evaluate(X_test_scaled, [y_test_age, y_test_hispanic, y_test_race, 
                                y_test_education, y_test_income, y_test_gender,
                                y_test_sexual_orientation, y_test_state, y_test_hh_size,
                                y_test_howmany, y_test_how_oft, y_test_cigarettes, 
                                y_test_marijuana, y_test_alcohol, y_test_diabetes,
                                y_test_wheelchair, y_test_life_changes], verbose=2, return_dict=True)

1356/1356 - 8s - 6ms/step - age_accuracy: 0.6743 - age_loss: 1.0636 - alcohol_accuracy: 0.7713 - alcohol_loss: 0.5923 - cigarettes_accuracy: 0.8509 - cigarettes_loss: 0.4346 - diabetes_accuracy: 0.9028 - diabetes_loss: 0.2199 - education_accuracy: 0.7458 - education_loss: 0.8440 - gender_accuracy: 0.7988 - gender_loss: 0.4993 - hh-size_accuracy: 0.7644 - hh-size_loss: 0.7811 - hispanic_accuracy: 0.9382 - hispanic_loss: 0.1848 - how-oft_accuracy: 0.7835 - how-oft_loss: 0.7133 - howmany_accuracy: 0.7120 - howmany_loss: 0.9066 - income_accuracy: 0.6314 - income_loss: 1.5702 - life-changes_accuracy: 0.7813 - life-changes_loss: 0.6985 - loss: 10.5203 - marijuana_accuracy: 0.8401 - marijuana_loss: 0.5178 - race_accuracy: 0.8678 - race_loss: 0.4365 - sexual-orientation_accuracy: 0.8536 - sexual-orientation_loss: 0.3995 - state_accuracy: 0.9034 - state_loss: 0.6137 - wheelchair_accuracy: 0.9871 - wheelchair_loss: 0.0427


In [65]:
print(f'age accuracy: {test_results['age_accuracy']}')
print(f'hispanic accuracy {test_results['hispanic_accuracy']}')
print(f'race accuracy {test_results['race_accuracy']}')
print(f'education accuracy {test_results['education_accuracy']}')
print(f'income accuracy {test_results['income_accuracy']}')
print(f'gender accuracy {test_results['gender_accuracy']}')
print(f'sexual orientation accuracy {test_results['sexual-orientation_accuracy']}')
print(f'state accuracy {test_results['state_accuracy']}')
print(f'howmany accuracy {test_results['howmany_accuracy']}')
print(f'hh-size accuracy {test_results['hh-size_accuracy']}')
print(f'how-oft accuracy {test_results['how-oft_accuracy']}')
print(f'cigarettes accuracy {test_results['cigarettes_accuracy']}')
print(f'marijuana accuracy {test_results['marijuana_accuracy']}')
print(f'alcohol accuracy {test_results['alcohol_accuracy']}')
print(f'diabetes accuracy {test_results['diabetes_accuracy']}')
print(f'wheelchair accuracy {test_results['wheelchair_accuracy']}')
print(f'life changes accuracy {test_results['life-changes_accuracy']}')

age accuracy: 0.6742754578590393
hispanic accuracy 0.9382075667381287
race accuracy 0.8678379654884338
education accuracy 0.7457517981529236
income accuracy 0.631435751914978
gender accuracy 0.7987595200538635
sexual orientation accuracy 0.8535657525062561
state accuracy 0.9033685922622681
howmany accuracy 0.7120426297187805
hh-size accuracy 0.7644047737121582
how-oft accuracy 0.7835419774055481
cigarettes accuracy 0.8509141802787781
marijuana accuracy 0.8401005268096924
alcohol accuracy 0.7712526917457581
diabetes accuracy 0.9028152227401733
wheelchair accuracy 0.9871112108230591
life changes accuracy 0.781328558921814


In [82]:
most_common_age_percent = (y_test['age'].value_counts(normalize=True).max()*100).round(2)
most_common_hispanic_percent = (y_test['hispanic'].value_counts(normalize=True).max()*100).round(2)
most_common_race_percent = (y_test['race'].value_counts(normalize=True).max()*100).round(2)
most_common_education_percent = (y_test['education'].value_counts(normalize=True).max()*100).round(2)
most_common_income_percent = (y_test['income'].value_counts(normalize=True).max()*100).round(2)
most_common_gender_percent = (y_test['gender'].value_counts(normalize=True).max()*100).round(2)
most_common_orientation_percent = (y_test['sexual-orientation'].value_counts(normalize=True).max()*100).round(2)
most_common_state_percent = (y_test['state'].value_counts(normalize=True).max()*100).round(2)
most_common_howmany_percent = (y_test['howmany'].value_counts(normalize=True).max()*100).round(2)
most_common_hh_percent = (y_test['hh-size'].value_counts(normalize=True).max()*100).round(2)
most_common_howoft_percent = (y_test['how-oft'].value_counts(normalize=True).max()*100).round(2)
most_common_cigarettes_percent = (y_test['cigarettes'].value_counts(normalize=True).max()*100).round(2)
most_common_marijuana_percent = (y_test['marijuana'].value_counts(normalize=True).max()*100).round(2)
most_common_alcohol_percent = (y_test['alcohol'].value_counts(normalize=True).max()*100).round(2)
most_common_diabetes_percent = (y_test['diabetes'].value_counts(normalize=True).max()*100).round(2)
most_common_wheelchair_percent = (y_test['wheelchair'].value_counts(normalize=True).max()*100).round(2)
most_common_changes_percent = (y_test['life-changes'].value_counts(normalize=True).max()*100).round(2)

print(f'Most common age occurs in {most_common_age_percent}% of rows')
print(f'Most common hispanic occurs in {most_common_hispanic_percent}% of rows')
print(f'Most common race occurs in {most_common_race_percent}% of rows')
print(f'Most common education occurs in {most_common_education_percent}% of rows')
print(f'Most common income occurs in {most_common_income_percent}% of rows')
print(f'Most common gender occurs in {most_common_gender_percent}% of rows')
print(f'Most common sexual orientation occurs in {most_common_age_percent}% of rows')
print(f'Most common state occurs in {most_common_state_percent}% of rows')
print(f'Most common howmany occurs in {most_common_howmany_percent}% of rows')
print(f'Most common hh-size occurs in {most_common_hh_percent}% of rows')
print(f'Most common how-oft occurs in {most_common_howoft_percent}% of rows')
print(f'Most common cigarette occurs in {most_common_cigarettes_percent}% of rows')
print(f'Most common marijuana occurs in {most_common_marijuana_percent}% of rows')
print(f'Most common alcohol occurs in {most_common_alcohol_percent}% of rows')
print(f'Most common diabetes occurs in {most_common_diabetes_percent}% of rows')
print(f'Most common wheelchair occurs in {most_common_wheelchair_percent}% of rows')
print(f'Most common life change occurs in {most_common_changes_percent}% of rows')


Most common age occurs in 30.6% of rows
Most common hispanic occurs in 91.41% of rows
Most common race occurs in 76.43% of rows
Most common education occurs in 42.45% of rows
Most common income occurs in 23.56% of rows
Most common gender occurs in 62.42% of rows
Most common sexual orientation occurs in 30.6% of rows
Most common state occurs in 8.9% of rows
Most common howmany occurs in 54.66% of rows
Most common hh-size occurs in 30.83% of rows
Most common how-oft occurs in 45.37% of rows
Most common cigarette occurs in 79.41% of rows
Most common marijuana occurs in 77.42% of rows
Most common alcohol occurs in 52.39% of rows
Most common diabetes occurs in 90.14% of rows
Most common wheelchair occurs in 98.77% of rows
Most common life change occurs in 71.11% of rows


In all variables, our model predicts demographics better than just predicting the most common case, except for wheelchair which predicts the as the most common case. This makes sense, as the percentage of the most common wheelchair value is over 98%.